# Pipeline project yeast


## 1. Trimming + BAM processing
Reads are filtered by strand ant trimmend to their last 30 nucleotides, then sorted and indexed



<small>Code Saif: 1. Trimming_pysam + indexing bam file</small>


In [1]:
!pip install biopython
!pip install pysam


import pysam
from Bio.Seq import Seq  # for reverse-complement

input_bam = "/home/saif/Documents/Saif_project_Yeast/Inputs/Saif/longReads/Scer.sorted.bam"
output_bam = "/home/ldufour/Documents/Project_Yeast/Trimming/output_reverseCorrected_last30.bam"
read_length = 30

with pysam.AlignmentFile(input_bam, "rb") as infile, \
     pysam.AlignmentFile(output_bam, "wb", template=infile) as outfile:

    for read in infile.fetch():
        if read.flag not in {0, 16}:
            continue  # keep only forward (0) and reverse (16)

        if read.query_length < read_length:
            continue

        # Get the aligned pairs
        aligned_pairs = read.get_aligned_pairs(matches_only=True)
        if len(aligned_pairs) < read_length:
            continue

        # Prepare new read object
        new_read = read.__copy__()

        if read.flag == 0:
            # Forward strand: last 30 nt from the 3' end
            trimmed_seq = read.query_sequence[-read_length:]
            trimmed_qual = read.query_qualities[-read_length:] if read.query_qualities else None
            new_start = aligned_pairs[-read_length][1]
            new_read.pos = new_start
        else:
            # Reverse strand: last 30 nt is from the 5' end (in read space), reverse complement it
            trimmed_seq = str(Seq(read.query_sequence[:read_length]).reverse_complement())
            if read.query_qualities:
                trimmed_qual = read.query_qualities[:read_length][::-1]
            else:
                trimmed_qual = None
            new_start = aligned_pairs[read_length - 1][1]  # start at the last of the first 30 aligned bases
            new_read.pos = new_start

        # Update the new read
        new_read.query_sequence = trimmed_seq
        new_read.query_qualities = trimmed_qual
        new_read.cigar = [(0, read_length)]  # 30M match
        outfile.write(new_read)

print("Output saved to:", output_bam)


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Output saved to: /home/ldufour/Documents/Project_Yeast/Trimming/output_reverseCorrected_last30.bam


In [3]:
%%bash 

module load SAMtools/1.19.2-GCC-13.3.0

samtools sort -o  /home/ldufour/Documents/Project_Yeast/Trimming/output_reverseCorrected_last30_sorted.bam \
/home/ldufour/Documents/Project_Yeast/Trimming/output_reverseCorrected_last30.bam

samtools index /home/ldufour/Documents/Project_Yeast/Trimming/output_reverseCorrected_last30_sorted.bam

[bam_sort_core] merging from 11 files and 1 in-memory blocks...


## 2. Peak calling
MACS2 is used to detect regions with high read accumulation, indicating potential transcript ends.

<small>Code Saif: 2. Peak Caller: MASC2</small>

In [4]:
%%bash
## peak controller. MASC2
module load macs2
macs2 callpeak \
-t /home/ldufour/Documents/Project_Yeast/Trimming/output_reverseCorrected_last30_sorted.bam \
--outdir /home/ldufour/Documents/Project_Yeast/Output/MASC2/ \
-n last30_2 \
--nomodel \
--extsize 30 \
--keep-dup auto \
-B --SPMR


The following have been reloaded with a version change:
  1) Python/3.12.3-GCCcore-13.3.0 => Python/3.8.0-GCCcore-13.3.0

INFO  @ Mon, 20 Apr 2026 08:34:18: 
# Command line: callpeak -t /home/ldufour/Documents/Project_Yeast/Trimming/output_reverseCorrected_last30_sorted.bam --outdir /home/ldufour/Documents/Project_Yeast/Output/MASC2/ -n last30_2 --nomodel --extsize 30 --keep-dup auto -B --SPMR
# ARGUMENTS LIST:
# name = last30_2
# format = AUTO
# ChIP-seq file = ['/home/ldufour/Documents/Project_Yeast/Trimming/output_reverseCorrected_last30_sorted.bam']
# control file = None
# effective genome size = 2.70e+09
# band width = 300
# model fold = [5, 50]
# qvalue cutoff = 5.00e-02
# The maximum gap between significant sites is assigned as the read length/tag size.
# The minimum length of peaks is assigned as the predicted fragment length "d".
# Larger dataset will be scaled towards smaller dataset.
# Range for calculating regional lambda is: 10000 bps
# Broad region calling is off
# Paire

## 3. New transcript annotation
New transcript boundaries are defined based on peaks and written to a GTF file.

<small> Code Saif: 3. Creating an annotation file for new transcripts</small>

In [5]:
!pip install gffutils

import pandas as pd
import gffutils
import os

# === File paths ===
GFF_FILE = "/home/saif/Documents/Saif_project_Yeast/Inputs/Saif/Annotation/Scer.utr.agat.gff"
NARROWPEAK_FILE = "/home/ldufour/Documents/Project_Yeast/Output/MASC2/last30_2_peaks.narrowPeak"
DB_FILE = "/home/ldufour/Documents/Project_Yeast/Output/New_annotation/narrowpeak_transcripts_trim30_2.db"
OUTPUT_GTF = "/home/ldufour/Documents/Project_Yeast/Output/New_annotation/narrowpeak_transcripts_trim30_2.gtf"

# === Load .narrowPeak and compute summit position ===
narrow_df = pd.read_csv(NARROWPEAK_FILE, sep='\t', header=None,
                        names=["chrom", "start", "end", "name", "score", "strand",
                               "signalValue", "pValue", "qValue", "peak"])

# Calculate summit position
narrow_df["summit_pos"] = narrow_df["start"] + narrow_df["peak"]
print(f"Total peaks loaded: {len(narrow_df)}")

# === Build or load GFF database ===
if not os.path.exists(DB_FILE):
    print("Creating GFF database...")
    gffutils.create_db(GFF_FILE, DB_FILE, merge_strategy="merge", keep_order=True)
db = gffutils.FeatureDB(DB_FILE)

# === Match summit positions to transcripts and build new entries ===
new_gtf_lines = []
seen = set()

for _, row in narrow_df.iterrows():
    chrom = row["chrom"]
    summit = int(row["summit_pos"])
    peak_name = row["name"]

    # Only get features where summit lies within transcript
    overlapping = db.region(region=(chrom, summit, summit + 1), completely_within=False)

    for feature in overlapping:
        if feature.featuretype not in ["mRNA", "transcript"]:
            continue

        if not (feature.start <= summit <= feature.end):
            continue

        gene_id = feature.attributes.get("gene_id", [None])[0] or feature.id
        strand = feature.strand

        if strand == "+":
            tx_start = feature.start
            tx_end = summit
        elif strand == "-":
            tx_start = summit
            tx_end = feature.end
        else:
            continue

        new_tx_id = f"{gene_id}_{peak_name}"
        if new_tx_id in seen:
            continue
        seen.add(new_tx_id)

        line = (
            f"{chrom}\tcustom\ttranscript\t{tx_start}\t{tx_end}\t.\t{strand}\t.\t"
            f'gene_id "{gene_id}"; transcript_id "{new_tx_id}";\n'
        )
        new_gtf_lines.append(line)

# === Write updated GTF ===
with open(OUTPUT_GTF, "w") as f:
    f.writelines(new_gtf_lines)

print(f"Updated transcript annotations saved to {OUTPUT_GTF}")

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Total peaks loaded: 8401
Creating GFF database...
Updated transcript annotations saved to /home/ldufour/Documents/Project_Yeast/Output/New_annotation/narrowpeak_transcripts_trim30_2.gtf


### 3.1 Making an annotation file with the longest new transcript/isoform
**Goal:** Keep only the longest new transcript per gene from the custom GTF.

In [19]:

import pandas as pd

def parse_gtf(file_path):
    entries = []
    with open(file_path, 'r') as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            parts = line.strip().split('\t')
            if len(parts) != 9 or parts[2] != "transcript":
                continue
            seq_id, source, feature_type, start, end, score, strand, phase, attributes = parts
            attr_dict = {}
            for attr in attributes.strip().split(';'):
                if ' ' in attr:
                    key, value = attr.strip().split(' ', 1)
                    attr_dict[key.strip()] = value.strip().strip('"')
            transcript_id = attr_dict.get('transcript_id', '')
            gene_id = attr_dict.get('gene_id', transcript_id.split('_')[0])
            entries.append({
                "line": line.strip(),
                "seq_id": seq_id,
                "start": int(start),
                "end": int(end),
                "length": int(end) - int(start),
                
                "strand": strand,
                "transcript_id": transcript_id,
                "gene_id": gene_id
            })
    return pd.DataFrame(entries)

# Paths
input_gtf = "/home/ldufour/Documents/Project_Yeast/Output/New_annotation/narrowpeak_transcripts_trim30_2.gtf"
output_gtf = "/home/ldufour/Documents/Project_Yeast/Output/New_annotation/trim30_2_longest_annotation.gtf"

# Parse the GTF
df = parse_gtf(input_gtf)

# Keep only the longest transcript per gene
longest_per_gene = df.sort_values(by="length", ascending=False).drop_duplicates(subset="gene_id")

# Write output
with open(output_gtf, "w") as f:
    for line in longest_per_gene['line']:
        f.write(line + "\n")

print(f"Longest transcript per gene saved to: {output_gtf}")

Longest transcript per gene saved to: /home/ldufour/Documents/Project_Yeast/Output/New_annotation/trim30_2_longest_annotation.gtf


## 4. CDS overlap analysis
The overlap between new transcripts and existing coding regions (CDS) is calculated.

### 4.1 Extraction of CDS Regions

In [7]:
#code saif: 6.1.1 Extract the CDS from Scer.utr.agat.gf
import pandas as pd

# === File paths ===
input_path = "/home/saif/Documents/Saif_project_Yeast/Inputs/Saif/Annotation/Scer.utr.agat.gff"
output_path = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/New_annotation/Scer.utr.CDS_only.gff"

# === Load the GFF file ===
gff = pd.read_csv(input_path, sep="\t", comment="#", header=None)

# === Filter only CDS rows ===
cds_df = gff[gff[2] == "CDS"]

# === Save to a new GFF file ===
cds_df.to_csv(output_path, sep="\t", header=False, index=False)

print(f"Extracted CDS entries saved to: {output_path}")

Extracted CDS entries saved to: /home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/New_annotation/Scer.utr.CDS_only.gff


### 4.2 Conversion of Anotation Files to BED Format

In [8]:
#Code Saif: 6.1.2 Convert the annotation files to bed files
import pandas as pd
import re

# === File paths ===
gtf_path = "/home/ldufour/Documents/Project_Yeast/Output/New_annotation/narrowpeak_transcripts_trim30_2.gtf"
cds_gff_path = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/New_annotation/Scer.utr.CDS_only.gff"
gtf_bed = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/for_cds_narrowpeak_transcripts_trim30_2.bed"
cds_bed = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/Scer.utr.CDS_only.bed"

# === Convert GTF to BED ===
gtf_df = pd.read_csv(gtf_path, sep="\t", comment='#', header=None,
    names=["seqid", "source", "feature", "start", "end", "score", "strand", "frame", "attribute"])
gtf_df = gtf_df[gtf_df["feature"] == "transcript"].copy()
gtf_df["transcript_id"] = gtf_df["attribute"].str.extract(r'transcript_id "([^"]+)"')

# Convert to BED format
gtf_df["start_bed"] = gtf_df["start"] - 1  # 0-based
gtf_bed_df = gtf_df[["seqid", "start_bed", "end", "transcript_id", "score", "strand"]].fillna(".")

# Save BED
gtf_bed_df.to_csv(gtf_bed, sep="\t", header=False, index=False)

# === Convert CDS GFF to BED ===
cds_df = pd.read_csv(cds_gff_path, sep="\t", comment='#', header=None,
    names=["seqid", "source", "feature", "start", "end", "score", "strand", "frame", "attribute"])
cds_df = cds_df[cds_df["feature"] == "CDS"].copy()
cds_df["cds_id"] = cds_df["attribute"].str.extract(r'gene_id=([^;]+)')
cds_df["start_bed"] = cds_df["start"] - 1
cds_bed_df = cds_df[["seqid", "start_bed", "end", "cds_id", "score", "strand"]].fillna(".")

# Save BED
cds_bed_df.to_csv(cds_bed, sep="\t", header=False, index=False)

print("GTF and CDS GFF successfully converted to BED.")

GTF and CDS GFF successfully converted to BED.


### 4.3 Identification of Overlaps Using BED Format

In [11]:

%%bash
#Code Saif: 6.1.3 BEDtools intersect
#module load BEDTools/2.31.1-GCC-13.3.0
module load BEDTools/2.31.1

# === File paths ===
cds_bed="/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/Scer.utr.CDS_only.bed"
transcripts_bed="/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/for_cds_narrowpeak_transcripts_trim30_2.bed"
output_dir="/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics"

#  1. Get overlapping transcripts with CDS and count overlapping nucleotides 
bedtools intersect \
  -a "$cds_bed" \
  -b "$transcripts_bed" \
  -s -wo \
  > "$output_dir/overlaps_transcript_cds_with_overlap_nt.bed"

#  2. Get non-overlapping transcripts 
bedtools intersect \
  -a "$transcripts_bed" \
  -b "$cds_bed" \
  -s -v \
  > "$output_dir/non_overlapping_transcripts.bed"

echo "Finished: Overlapping and non-overlapping transcripts extracted."

Finished: Overlapping and non-overlapping transcripts extracted.


### 4.4 Filtering Overlqps by Gene Consistency
Filter the BED file to retain only transcripts that are derived from their annotated parent genes.

In [12]:
import pandas as pd

# === Load the BED file ===
input_file = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/overlaps_transcript_cds_with_overlap_nt.bed"
output_file = "//home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/filtered_transcript_cds_overlap.bed"

# No headers in BED, so specify them
cols = [
    "cds_chr", "cds_start", "cds_end", "gene_id", "cds_score", "cds_strand",
    "tx_chr", "tx_start", "tx_end", "transcript_id", "tx_score", "tx_strand",
    "overlap_nt"
]

df = pd.read_csv(input_file, sep="\t", header=None, names=cols)

# === Extract gene name from transcript_id by splitting at the first underscore ===
df["tx_gene"] = df["transcript_id"].str.split("_", n=1).str[0]

# === Keep only rows where gene_id matches tx_gene ===
df_filtered = df[df["gene_id"] == df["tx_gene"]]

# === Save the filtered output ===
df_filtered.to_csv(output_file, sep="\t", header=False, index=False)

print(f"Filtered file saved: {output_file}")
print(f"Kept {len(df_filtered)} out of {len(df)} rows.")

Filtered file saved: //home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/filtered_transcript_cds_overlap.bed
Kept 8887 out of 9347 rows.


### 4.5 Merging Overlaps per Transcript 
Treat overlapping with two CDS regions of the same gene as a single overlap for the transcript.

In [13]:
import pandas as pd

# === Load your overlap file ===
input_file = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/filtered_transcript_cds_overlap.bed"
output_file = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/merged_overlap_counts.bed"

# Define column names (modify if your file structure differs)
cols = [
    "cds_chr", "cds_start", "cds_end", "gene_id", "cds_score", "cds_strand",
    "tx_chr", "tx_start", "tx_end", "transcript_id", "tx_score", "tx_strand",
    "overlap_nt", "gene_match"
]

# Read the file
df = pd.read_csv(input_file, sep="\t", header=None, names=cols)

# === Group by gene_id and transcript_id, and sum the overlaps ===
df_grouped = df.groupby(["gene_id", "transcript_id"], as_index=False)["overlap_nt"].sum()

# === Optional: sort by gene or overlap size ===
df_grouped = df_grouped.sort_values(by=["gene_id", "overlap_nt"], ascending=[True, False])

# === Save result ===
df_grouped.to_csv(output_file, sep="\t", header=True, index=False)

print(f"Saved grouped overlaps to: {output_file}")

Saved grouped overlaps to: /home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/merged_overlap_counts.bed


### 4.6 Calculation of CDS Length per Transcript

In [14]:
import pandas as pd

# Input and Output File Paths
input_file = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/filtered_transcript_cds_overlap.bed"  
output_file = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/cds_length_per_transcript.csv"

# Define columns based on your example format
cols = [
    "cds_chr", "cds_start", "cds_end", "gene_id", "cds_score", "cds_strand",
    "tx_chr", "tx_start", "tx_end", "transcript_id", "tx_score", "tx_strand",
    "overlap_nt", "gene_match"
]

# Load the data
df = pd.read_csv(input_file, sep="\t", header=None, names=cols)

# Calculate CDS length per row (inclusive range)
df["cds_length"] = df["cds_end"] - df["cds_start"]

# Sum CDS length per transcript
cds_length_per_transcript = df.groupby("transcript_id")["cds_length"].sum().reset_index()

# Save to CSV
cds_length_per_transcript.to_csv(output_file, index=False)

print(f"CDS lengths per transcript saved to: {output_file}")

CDS lengths per transcript saved to: /home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/cds_length_per_transcript.csv


### 4.7 Generation of Initial Summary Table

In [15]:
import pandas as pd

# === File paths ===
merged_overlap_path = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/merged_overlap_counts.bed"  
non_overlap_path = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/non_overlapping_transcripts.bed"  
output_csv_path = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/summary.csv"

# === Load merged overlaps ===
# Expected columns: gene_id, transcript_id, overlap_nt
df_overlap = pd.read_csv(merged_overlap_path, sep="\t")

# === Load non-overlapping transcripts ===
# Expected BED columns: chr, start, end, transcript_id, score, strand
df_non_overlap = pd.read_csv(non_overlap_path, sep="\t", header=None)
df_non_overlap.columns = ["chr", "start", "end", "transcript_id", "score", "strand"]

# === Extract gene_id from transcript_id (part before first underscore) ===
df_non_overlap["gene_id"] = df_non_overlap["transcript_id"].str.split("_", n=1).str[0]

# === Create DataFrame with overlap_nt = 0 ===
df_non_overlap_final = df_non_overlap[["gene_id", "transcript_id"]].copy()
df_non_overlap_final["overlap_nt"] = 0

# === Combine both DataFrames ===
df_combined = pd.concat([df_overlap, df_non_overlap_final], ignore_index=True)

# === Save to CSV ===
df_combined.to_csv(output_csv_path, index=False)

print(f"Combined overlap summary saved to: {output_csv_path}")

Combined overlap summary saved to: /home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/summary.csv


### 4.8 Integration of CDS Length into Summary

In [16]:
import pandas as pd

# === File Paths ===
summary_path = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/summary.csv"  
cds_length_path = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/cds_length_per_transcript.csv"  

# === Load files ===
df_summary = pd.read_csv(summary_path)
df_cds_length = pd.read_csv(cds_length_path)

# === Merge CDS lengths into the summary ===
df_summary = pd.merge(df_summary, df_cds_length, on="transcript_id", how="left")

# === Overwrite the existing summary file ===
df_summary.to_csv(summary_path, index=False)

print(f"CDS lengths successfully added to existing summary file: {summary_path}")

CDS lengths successfully added to existing summary file: /home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/summary.csv


### 4.9 Calculation of CDS Overlap Percentages


In [17]:
import pandas as pd

# === Input and Output File ===
input_csv = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/summary.csv"  


# === Load the summary data ===
df = pd.read_csv(input_csv)

# === Calculate percent overlap ===
df["overlap_pct"] = (df["overlap_nt"] / df["cds_length"]) * 100

# === Save back to the same file (overwrite) ===
df.to_csv(input_csv, index=False)

# === Optional: save to a different file ===
# df.to_csv(output_csv, index=False)

print(f"Percent overlap added to: {input_csv}")

Percent overlap added to: /home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/summary.csv


### 4.10 Identification of Longest Isoforms

In [20]:
#code Saif: 6.1.5 Add a flag indicating whether the transcript is the longest isoform
import pandas as pd

# === File paths ===
overlap_csv = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/summary.csv"
longest_gtf = "/home/ldufour/Documents/Project_Yeast/Output/New_annotation/trim30_2_longest_annotation.gtf"
output_csv = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/summary_cds_transcript_overlap_percent_FLAGGED_FIXED.csv"

# === Step 1: Load summary/overlap data ===
overlap_df = pd.read_csv(overlap_csv)
overlap_df["transcript_id"] = overlap_df["transcript_id"].astype(str).str.strip()

# === Step 2: Load GTF and extract transcript_id ===
gtf_df = pd.read_csv(longest_gtf, sep="\t", comment="#", header=None)

# Extract transcript_id from attributes column (usually column 8)
# Adjust if it's in a different column
gtf_df["transcript_id"] = gtf_df[8].astype(str).str.extract(r'transcript_id[ ="]*([^";]+)')

# === Step 3: Build set of longest transcript IDs ===
longest_ids = set(gtf_df["transcript_id"].dropna().unique())

# === Step 4: Flag overlap rows ===
overlap_df["is_longest"] = overlap_df["transcript_id"].isin(longest_ids)

# === Step 5: Save result ===
overlap_df.to_csv(output_csv, index=False)
print(f"Summary file with 'is_longest' flag saved to:\n{output_csv}")

Summary file with 'is_longest' flag saved to:
/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/summary_cds_transcript_overlap_percent_FLAGGED_FIXED.csv


### 4.11 Addition of Transcript Length and Strand Information

In [21]:
import pandas as pd
import re

# === File paths ===
summary_path = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/summary_cds_transcript_overlap_percent_FLAGGED_FIXED.csv"
gtf_path = "/home/ldufour/Documents/Project_Yeast/Output/New_annotation/narrowpeak_transcripts_trim30_2.gtf"
output_path = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/summary1_with_transcript_length_strand.csv"

# === Load summary file and GTF ===
summary_df = pd.read_csv(summary_path)
gtf_df = pd.read_csv(gtf_path, sep="\t", comment='#', header=None)

# === Filter GTF for transcript entries only ===
gtf_df = gtf_df[gtf_df[2] == "transcript"].copy()

# === Extract transcript_id from attributes ===
gtf_df["transcript_id"] = gtf_df[8].str.extract(r'transcript_id "([^"]+)"')

# === Calculate transcript length and extract strand ===
gtf_df["transcript_length"] = gtf_df[4] - gtf_df[3] + 1
gtf_df["transcript_strand"] = gtf_df[6]

# === Build minimal DataFrame for merging ===
lengths_strands = gtf_df[["transcript_id", "transcript_length", "transcript_strand"]]

# === Merge into summary ===
merged_df = summary_df.merge(lengths_strands, on="transcript_id", how="left")

# === Save output ===
merged_df.to_csv(output_path, index=False)
print(f"summary1 saved to: {output_path}")

summary1 saved to: /home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/summary1_with_transcript_length_strand.csv


## 5. Strand-Aware Peak Quantification

### 5.1 Strand Assignment by Peak Quantification
Duplicate each peak from the original .narrowPeak file to create two versions: one assuming + strand and one - strand. This is necessary because narrowPeak files often lack strand information, but strand-specific counting tools (like featureCounts -s 1) require it.

In [22]:
#Code Saif: 6.2.1 Extend Peaks to Both Strands
import pandas as pd

# === File paths ===
input_path = "/home/ldufour/Documents/Project_Yeast/Output/MASC2/last30_2_peaks.narrowPeak"
output_path = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/last30_2_peaks_with_strands.narrowPeak"

# === Load narrowPeak file ===
# narrowPeak has 10 standard columns; column 5 is strand (we'll overwrite it)
peaks_df = pd.read_csv(input_path, sep="\t", header=None)

# === Duplicate peaks with strand set to "+" and "-"
peaks_plus = peaks_df.copy()
peaks_minus = peaks_df.copy()

peaks_plus[5] = "+"
peaks_minus[5] = "-"

# === Combine both strand entries
extended_df = pd.concat([peaks_plus, peaks_minus], ignore_index=True)

# === Save to file
extended_df.to_csv(output_path, sep="\t", header=False, index=False)
print(f"Peaks with both strand orientations saved to: {output_path}")

Peaks with both strand orientations saved to: /home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/last30_2_peaks_with_strands.narrowPeak


### 5.2 Conversion of Peaks to SAF format
Transform the strand-duplicated .narrowPeak file into SAF (Simplified Annotation Format) for compatibility with featureCounts.

In [24]:
#Code Saif: 6.2.2 Convert Peaks to SAF Format
import pandas as pd

# Input narrowPeak file
peaks_file = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/last30_2_peaks_with_strands.narrowPeak"
saf_file = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/last30_2_peaks_with_strands.saf"

# Load narrowPeak
df = pd.read_csv(peaks_file, sep="\t", header=None)

# Build SAF format
# SAF: GeneID, Chr, Start, End, Strand
df_saf = pd.DataFrame({
    "GeneID": df[3],         # peak name
    "Chr": df[0],
    "Start": df[1] + 1,      # SAF uses 1-based start
    "End": df[2],
    "Strand": df[5]
})

# Save SAF
df_saf.to_csv(saf_file, sep="\t", index=False)
print(f"SAF file created at: {saf_file}")

SAF file created at: /home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/last30_2_peaks_with_strands.saf


### 5.3 Strand-Specifc Read Counting (featureCounts)
After counting, modify the GeneIDs in the SAF file to explicitly tag them as **_positive** or **_negative**, making it easy to interpret counts.

In [25]:
#Code Saif: 6.2.4 Disambiguate Peak IDs  + run featureCounts again 
import pandas as pd

# === File paths ===
input_saf = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/last30_2_peaks_with_strands.saf"
output_saf = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/last30_2_peaks_with_strands_stranded_ids.saf"

# === Load SAF file ===
saf_df = pd.read_csv(input_saf, sep="\t")

# === Modify GeneID based on strand
saf_df["GeneID"] = saf_df.apply(
    lambda row: f"{row['GeneID']}_positive" if row["Strand"] == "+"
    else f"{row['GeneID']}_negative" if row["Strand"] == "-"
    else row["GeneID"],
    axis=1
)

# === Save updated SAF
saf_df.to_csv(output_saf, sep="\t", index=False)
print(f"Stranded SAF file saved to: {output_saf}")

Stranded SAF file saved to: /home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/last30_2_peaks_with_strands_stranded_ids.saf


In [26]:
%%bash

# Activate conda environment
source /home/saif/anaconda3/bin/activate featurecounts

# Run featureCounts with strand-tagged SAF
featureCounts \
  -a /home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/last30_2_peaks_with_strands_stranded_ids.saf \
  -F SAF \
  -o /home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/read_counts_per_peak_stranded.txt \
  -T 20 \
  -s 1 \
  /home/ldufour/Documents/Project_Yeast/Trimming/output_reverseCorrected_last30_sorted.bam


        ==========     _____ _    _ ____  _____  ______          _____  
        =====         / ____| |  | |  _ \|  __ \|  ____|   /\   |  __ \ 
          =====      | (___ | |  | | |_) | |__) | |__     /  \  | |  | |
            ====      \___ \| |  | |  _ <|  _  /|  __|   / /\ \ | |  | |
              ====    ____) | |__| | |_) | | \ \| |____ / ____ \| |__| |
        ==========   |_____/ \____/|____/|_|  \_\______/_/    \_\_____/
	  v2.0.1

//========================== featureCounts setting ===========================\\
||                                                                            ||
||             Input files : 1 BAM file                                       ||
||                           o output_reverseCorrected_last30_sorted.bam      ||
||                                                                            ||
||             Output file : read_counts_per_peak_stranded.txt                ||
||                 Summary : read_counts_per_peak_stranded.txt.su

### 5.4 Strand Bias Analysis

You analyze the counts for each original peak (now represented by two entries: one for each strand) and:

- Compute the percentage of reads aligning to each strand.
- Classify peaks as:
  - **positive**: ≥80% of reads on the + strand
  - **negative**: ≥80% on the - strand
  - **both**: 20–80% on both strands
  - **discard**: <20% on both strands

In [27]:
#Code Saif: 6.2.5 Summarize Strand Bias per Peak
import pandas as pd

# === File path ===
counts_file = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/read_counts_per_peak_stranded.txt"

# === Load featureCounts output ===
df = pd.read_csv(counts_file, sep="\t", comment="#")

# Rename for clarity (assumes 1 sample column)
df.columns = ["GeneID", "Chr", "Start", "End", "Strand", "Length", "Count"]

# Extract base peak ID (without _positive/_negative)
df["BasePeakID"] = df["GeneID"].str.replace(r"_positive|_negative", "", regex=True)
df["StrandType"] = df["GeneID"].str.extract(r"_(positive|negative)")

# Pivot to get positive/negative counts side-by-side
pivot_df = df.pivot(index="BasePeakID", columns="StrandType", values="Count").fillna(0)

# Calculate percentages
pivot_df["Total"] = pivot_df["positive"] + pivot_df["negative"]
pivot_df["Positive_pct"] = (pivot_df["positive"] / pivot_df["Total"]) * 100
pivot_df["Negative_pct"] = (pivot_df["negative"] / pivot_df["Total"]) * 100

# Optional: Save to CSV
output_csv = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/strand_bias_summary_per_peak.csv"
pivot_df.to_csv(output_csv)

print(f"Strand bias summary saved to: {output_csv}")



Strand bias summary saved to: /home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/strand_bias_summary_per_peak.csv


### 5.5 Filtering of Strand-Assigned Peaks

In [28]:
#### filtering #### 

import pandas as pd

# === Load the strand bias summary ===
input_csv = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/strand_bias_summary_per_peak.csv"
df = pd.read_csv(input_csv)

# === Apply classification
def classify_strand(row):
    if row["positive"] + row["negative"] == 0:
        return None  # exclude empty peaks
    elif row["Positive_pct"] >= 80:
        return "positive"
    elif row["Negative_pct"] >= 80:
        return "negative"
    elif 20 < row["Positive_pct"] < 80:
        return "both"
    else:
        return None  # discard if both are < 20%

# Add new classification column
df["Assigned_Strand"] = df.apply(classify_strand, axis=1)

# Filter out rows with no assignment
filtered_df = df[df["Assigned_Strand"].notna()].copy()

# === Save filtered output
output_filtered = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/filtered_stranded_peaks.csv"
filtered_df.to_csv(output_filtered, index=False)

print(f"Filtered peaks saved to: {output_filtered}")


Filtered peaks saved to: /home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/filtered_stranded_peaks.csv


### 5.6 Expansion of Ambiguous Strand Peaks

In [29]:
#### Expand both-strand Peaks ####
## Since featureCounts cannot annotate "both",
## you duplicate "both" rows into separate positive and negative versions,
## ensuring each peak is strand-specific for downstream tools.##

import pandas as pd

# === Input and output paths ===
input_csv = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/filtered_stranded_peaks.csv"
output_csv = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/filtered_stranded_peaks_expanded.csv"

# === Load filtered peak summary
df = pd.read_csv(input_csv)

# === Split 'both' rows into two entries
both_df = df[df["Assigned_Strand"] == "both"].copy()
positive_df = df[df["Assigned_Strand"] == "positive"].copy()
negative_df = df[df["Assigned_Strand"] == "negative"].copy()

# Duplicate 'both' into positive and negative
both_pos = both_df.copy()
both_pos["Assigned_Strand"] = "positive"

both_neg = both_df.copy()
both_neg["Assigned_Strand"] = "negative"

# Combine all
final_df = pd.concat([positive_df, negative_df, both_pos, both_neg], ignore_index=True)

# Save result
final_df.to_csv(output_csv, index=False)
print(f"Expanded stranded peaks saved to: {output_csv}")

Expanded stranded peaks saved to: /home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/filtered_stranded_peaks_expanded.csv


### 5.7 Final Annotation of Strand-Specific Peaks

In [30]:
#### Tag Final Output
## You produce a clean, 
## tagged file with each peak labeled by strand (_positive, _negative, or _bothpositive/_bothnegative),
## suitable for plotting, filtering, or further analysis.

import pandas as pd

# === File paths ===
input_csv = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/filtered_stranded_peaks.csv"
output_csv = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/filtered_stranded_peaks_expanded_and_tagged.csv"

# === Load data
df = pd.read_csv(input_csv)

# Use the base ID as starting GeneID
df["GeneID"] = df["BasePeakID"]

# === Split rows
both_df = df[df["Assigned_Strand"] == "both"].copy()
positive_df = df[df["Assigned_Strand"] == "positive"].copy()
negative_df = df[df["Assigned_Strand"] == "negative"].copy()

# Expand "both" into two rows and tag GeneID
both_pos = both_df.copy()
both_pos["Assigned_Strand"] = "positive"
both_pos["GeneID"] = both_pos["GeneID"] + "_bothpositive"

both_neg = both_df.copy()
both_neg["Assigned_Strand"] = "negative"
both_neg["GeneID"] = both_neg["GeneID"] + "_bothnegative"

# Tag others normally
positive_df["GeneID"] = positive_df["GeneID"] + "_positive"
negative_df["GeneID"] = negative_df["GeneID"] + "_negative"

# Combine all
final_df = pd.concat([positive_df, negative_df, both_pos, both_neg], ignore_index=True)

# Save result
final_df.to_csv(output_csv, index=False)
print(f"Tagged peak file saved to: {output_csv}")

Tagged peak file saved to: /home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/filtered_stranded_peaks_expanded_and_tagged.csv


## 6. Integration of Read Counts info Summary Table

    Merged strand-aware read counts into the summary table summary1_with_transcript_length_strand.csv.

    Matched peaks by transcript strand and ID.

In [31]:
import pandas as pd

# === File paths ===
summary_path = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/summary1_with_transcript_length_strand.csv"
filtered_path = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/filtered_stranded_peaks_expanded_and_tagged.csv"
output_path = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/summary1_with_correct_read_counts.csv"

# === Load data ===
summary_df = pd.read_csv(summary_path)
filtered_df = pd.read_csv(filtered_path)

# === Extract BasePeakID from transcript_id ===
summary_df["BasePeakID"] = summary_df["transcript_id"].str.extract(r'^[^_]+_(last30_2_peak_\d+)', expand=False)

# === Define function to match read counts using partial GeneID ===
def get_read_count(row):
    base_peak_id = row["BasePeakID"]
    strand = row["transcript_strand"]
    
    # Find matching rows that start with the base peak ID
    matched = filtered_df[filtered_df["GeneID"].str.startswith(base_peak_id)]
    
    if matched.empty:
        return 0
    
    if strand == "+":
        return matched.iloc[0]["positive"]
    elif strand == "-":
        return matched.iloc[0]["negative"]
    
    return 0

# === Apply logic ===
summary_df["read_counts"] = summary_df.apply(get_read_count, axis=1)

# === Save the updated summary ===
summary_df.to_csv(output_path, index=False)
print(f"summary1 saved with correct read counts to: {output_path}")

summary1 saved with correct read counts to: /home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/summary1_with_correct_read_counts.csv


# Problem solve?

In [ ]:
import pandas as pd

# === File paths ===
summary_path = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/summary1_with_transcript_length_strand.csv"
filtered_path = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/filtered_stranded_peaks_expanded_and_tagged.csv"
output_path = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/summary1_with_correct_read_counts_solve.csv"

# === Load data ===
summary_df = pd.read_csv(summary_path)
filtered_df = pd.read_csv(filtered_path)

# === Extract BasePeakID from transcript_id ===
summary_df["BasePeakID"] = summary_df["transcript_id"].str.extract(r'^[^_]+_(last30_2_peak_\d+)', expand=False)

# === Define function to match read counts using partial GeneID ===
def get_read_count(row):
    base_peak_id = row["BasePeakID"]
    strand = row["transcript_strand"]

    if strand == "+":
        gene_id = base_peak_id + "_positive"
    elif strand == "-":
        gene_id = base_peak_id + "_negative"
    else:
        return 0

    matched = filtered_df[filtered_df["GeneID"] == gene_id]

    if matched.empty:
        # fallback: try bothpositive/bothnegative
        if strand == "+":
            gene_id = base_peak_id + "_bothpositive"
        elif strand == "-":
            gene_id = base_peak_id + "_bothnegative"
        matched = filtered_df[filtered_df["GeneID"] == gene_id]

    if matched.empty:
        return 0

    if strand == "+":
        return matched.iloc[0]["positive"]
    elif strand == "-":
        return matched.iloc[0]["negative"]
    return 0

# === Apply logic ===
summary_df["read_counts"] = summary_df.apply(get_read_count, axis=1)

# === Save the updated summary ===
summary_df.to_csv(output_path, index=False)
print(f"summary1 saved with correct read counts to: {output_path}")

summary1 saved with correct read counts to: /home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/summary1_with_correct_read_counts_solve.csv


## 7. Read-Peak-Transcript Linking

### 7.1 Conversion of SAF to BED Format

Convert a SAF file with stranded peaks to BED format for intersection analysis

In [32]:
#Code Saif: 7.5.1 Convert SAF to BED
import pandas as pd

# Load SAF
saf_df = pd.read_csv("/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/last30_2_peaks_with_strands_stranded_ids.saf", sep="\t")

# Convert to BED
bed_df = pd.DataFrame({
    "chr": saf_df["Chr"],
    "start": saf_df["Start"] - 1,  # BED is 0-based
    "end": saf_df["End"],
    "name": saf_df["GeneID"],
    "score": 0,
    "strand": saf_df["Strand"]
})

bed_df.to_csv("/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/peaks_stranded_saf.bed", sep="\t", header=False, index=False)

### 7.2 Extraction of Transcript Coordinates (GTF --> BED)

In [33]:
%%bash

awk '$3 == "transcript" {
    for (i=1; i<=NF; i++) {
        if ($i == "transcript_id") {
            gsub(/"/, "", $(i+1));
            transcript_id = $(i+1);
        }
    }
    print $1"\t"($4-1)"\t"$5"\t"transcript_id"\t0\t"$7;
}' /home/ldufour/Documents/Project_Yeast/Output/New_annotation/narrowpeak_transcripts_trim30_2.gtf \
> /home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/all_transcripts.bed

### 7.3 Linking Peaks to Transcripts

In [34]:

%%bash
#Code Saif: 7.6.3
module load BEDTools/2.31.1

bedtools intersect \
  -a /home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/peaks_stranded_saf.bed \
  -b /home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/all_transcripts.bed \
  -s -wa -wb \
  > /home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/peaks_with_transcript_info.bed

### 7.4 Linking Reads to Peaks and Transcript

In [35]:
%%bash

module load BEDTools/2.31.1

bedtools intersect \
  -a /home/ldufour/Documents/Project_Yeast/Trimming/output_reverseCorrected_last30_sorted.bam \
  -b /home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/peaks_with_transcript_info.bed \
  -bed -s -wa -wb \
  > /home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/reads_peaks_transcripts_with_txinfo.bed

### 7.5 Extraction of Read Names per Transcript/Peak

In [36]:
import pandas as pd

# === File Paths ===
input_path = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/reads_peaks_transcripts_with_txinfo.bed"
output_path = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/read_names_per_peak_transcript_filtered_improved.tsv"

# === Column Names (based on bedtools intersect output)
columns = [
    "read_chr", "read_start", "read_end", "read_id", "score", "strand",
    "thick_start", "thick_end", "item_rgb", "block_count", "block_sizes", "block_starts",
    "peak_chr", "peak_start", "peak_end", "peak_id", "peak_score", "peak_strand",
    "tx_chr", "tx_start", "tx_end", "transcript_id", "tx_score", "tx_strand"
]

# === Load Data ===
df = pd.read_csv(input_path, sep="\t", header=None, names=columns)

# === Extract base peak and transcript IDs
df["peak_base"] = df["peak_id"].str.extract(r"(last30_2_peak_\d+)")
df["transcript_base"] = df["transcript_id"].str.extract(r"(last30_2_peak_\d+)")
df["gene_id"] = df["transcript_id"].str.extract(r"(^[\w\d]+)")  # Extract gene name from transcript_id

# === Filter: Keep only exact peak-transcript matches
df_filtered = df[df["peak_base"] == df["transcript_base"]]

# === Group by transcript and peak, collect read names and count
grouped = df_filtered.groupby(["gene_id", "transcript_id", "peak_id"])["read_id"].agg(
    read_names=lambda x: ";".join(sorted(set(x))),  # Join read IDs in one line
    read_count="count"
).reset_index()

# === Save Output
grouped.to_csv(output_path, sep="\t", index=False)

print(f" Done. Saved filtered read names per peak/transcript to:\n{output_path}")


 Done. Saved filtered read names per peak/transcript to:
/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/read_names_per_peak_transcript_filtered_improved.tsv


## Poly(A) Tail Analysis

### 8.1 Poly(A) Tail Length Calculation per Transcript
For each transcript (or gene), find which reads overlap with poly(A) tail length data, and compute the median poly(A) length = poly(A) tail length quantification for transcript-specific mapped reads based on your custom annotation.

To compute poly(A) tail statistics (specifically the median length) for each transcript/peak combination, using only high-quality reads (qc_tag == PASS).

In [37]:
#Code Saif: 7.6.6 calculate the median poly(A) tail length per transcript or gene, based on overlapping reads
import pandas as pd

# === File Paths ===
reads_path = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/read_names_per_peak_transcript_filtered_improved.tsv"
polya_path = "/home/saif/Documents/Saif_project_Yeast/Inputs/Saif/longReads/polya/Scer.line.5.tsv"
output_path = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/polya_stats_all.tsv"

# === Load Read Names Data ===
reads_df = pd.read_csv(reads_path, sep="\t", dtype=str)
reads_df["transcript_id"] = reads_df["transcript_id"].str.strip().str.replace(";", "", regex=False)

# === Load PolyA Data and skip malformed rows ===
cols = [
    "readname", "contig", "position", "leader_start", "adapter_start",
    "polya_start", "transcript_start", "read_rate", "polya_length", "qc_tag"
]
polya_df = pd.read_csv(polya_path, sep="\t", names=cols, header=0, on_bad_lines='skip')

# === Keep only PASS entries
polya_df = polya_df[polya_df["qc_tag"] == "PASS"]

# === Initialize result container
results = []

# === Iterate through each row
for idx, row in reads_df.iterrows():
    gene_id = row["gene_id"]
    transcript_id = row["transcript_id"]
    peak_id = row["peak_id"]
    
    read_names = row["read_names"].split(";")
    read_names = [r.strip() for r in read_names if r.strip()]
    
    matching_polya = polya_df[polya_df["readname"].isin(read_names)]
    
    if not matching_polya.empty:
        median_polya = round(matching_polya["polya_length"].median(), 2)
        num_reads_with_polya = matching_polya.shape[0]
    else:
        median_polya = -1
        num_reads_with_polya = 0
    
    total_mapped_reads = len(read_names)
    
    results.append({
        "gene_id": gene_id.split("_")[0],  # Extract only gene name like Q0045
        "transcript_id": transcript_id,
        "peak_id": peak_id,
        "median_polya_length": median_polya,
        "num_reads_with_polya": num_reads_with_polya,
        "total_mapped_reads": total_mapped_reads
    })

# === Save Output
results_df = pd.DataFrame(results)
results_df.to_csv(output_path, sep="\t", index=False)

print(f"Saved polyA stats for all transcripts to:\n{output_path}")


Saved polyA stats for all transcripts to:
/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/polya_stats_all.tsv


### 8.2 Integration of Poly(A) Statistics into Summary

In [2]:
#Code Saif: 7.6.7 To add median_polya_length and num_reads_with_polya from polya_stats_all.tsv to summary1_with_correct_read_counts.csv, you can perform a merge on gene_id and transcript_id.
import pandas as pd

# === File Paths ===
polya_stats_path = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/polya_stats_all.tsv"
summary_path = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/summary1_with_correct_read_counts_solve.csv"
output_path = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/summary1_with_polya_stats_solve.csv"

# === Load Data ===
polya_df = pd.read_csv(polya_stats_path, sep="\t")
summary_df = pd.read_csv(summary_path)

# === Merge only on transcript_id
merged_df = pd.merge(
    summary_df,
    polya_df[["transcript_id", "median_polya_length", "num_reads_with_polya"]],
    on="transcript_id",
    how="left"
)

# === Fill missing polyA values with 0
merged_df[["median_polya_length", "num_reads_with_polya"]] = merged_df[["median_polya_length", "num_reads_with_polya"]].fillna(0)

# === Optional: convert types
merged_df["num_reads_with_polya"] = merged_df["num_reads_with_polya"].astype(int)

# === Save result
merged_df.to_csv(output_path, index=False)

print(f"Saved merged summary with polyA stats to:\n{output_path}")


Saved merged summary with polyA stats to:
/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/summary1_with_polya_stats_solve.csv


## 9. alt-TTS en std-TTS

### 9.1 Creating peaks_std-TTS.txt

In [3]:
import pandas as pd

input_file = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/summary1_with_polya_stats_solve.csv"
output_file = "/home/ldufour/Documents/Project_Yeast/Output/TTS/peaks_ID_std-TTS.txt"

df = pd.read_csv(input_file)

std_TTS_df = df[
    (df['is_longest'] == True)]


BasePeakID = std_TTS_df["BasePeakID"]

BasePeakID.to_csv(output_file, index=False, header=False)

### 9.2 Eliminate false positive cases

In [4]:
# by Mar Albà 16 Sep 2025

print("This program will select alt-TTS that do not coincide with std-TTS in file summary1_with_polya_stats.csv generated by Saif")
print("It requires a file with a list of peaks for std-TTS")


# read list of files

fh=open("/home/ldufour/Documents/Project_Yeast/Output/TTS/peaks_ID_std-TTS.txt",'r')  
text=fh.readlines()
fh.close()
list_std=[]
for line in text:
    peak=line.replace('\n','')
    list_std.append(peak)


fh=open("/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/summary1_with_polya_stats_solve.csv",'r')
text=fh.readlines()
fh.close()
selected=[]
for line in text:
    col = line.split(",")
    peak = col [8]
    std_tts = col [5]
    redundant=0
    for n in range(len(list_std)):
        if ((std_tts=="False") and (peak==list_std[n])):  # cases in which peak is labeled as alt-TTS but it is equal to an std-TTS peak
            redundant=1
            break
    if (redundant == 0):
        selected.append(line)
print(len(selected))

table_results="/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/summary1_with_polya_stats_clean_solve.csv"
fr=open(table_results, 'w')
for n in range(len(selected)):
    fr.write(selected[n])
fr.close() 



This program will select alt-TTS that do not coincide with std-TTS in file summary1_with_polya_stats.csv generated by Saif
It requires a file with a list of peaks for std-TTS
8829


In [5]:

import pandas as pd

# === File Paths ===
input_path  = "/home/ldufour/Documents/Project_Yeast/Output/Quantification_Statistics/summary1_with_polya_stats_clean_solve.csv"
alt_tts_path = "/home/ldufour/Documents/Project_Yeast/Output/TTS/Alt-TTS_solve.csv"
std_tts_path = "/home/ldufour/Documents/Project_Yeast/Output/TTS/Std-TTS_solve.csv"

# === Load Data ===
df = pd.read_csv(input_path)


# === Split into alt-TTS and std-TTS based on is_longest ===
alt_tts = df[
    (df['is_longest'] == False) &
    (df['read_counts'] >= 50 ) &
    (df['transcript_length'] > 50) &
    (df['transcript_length'] < 600 ) &
    (df['overlap_pct'] < 10) &
    ~ ((df['overlap_nt'] == 0) & (df['transcript_length'] > 250)) &
    ~df['gene_id'].str.startswith('agat')] 
std_tts  = df[
    (df['is_longest'] == True) &
    (df['read_counts'] >= 50)]

# === Rename columns ===
rename_dict = {
    "overlap_nt": "overlap_cds",
    "cds_length": "cds_length_when_overlap",
    "overlap_pct": "%_overlap_with_cds"}

alt_tts = alt_tts.rename(columns=rename_dict)
std_tts = std_tts.rename(columns=rename_dict)

# === Drop columns ===
cols_to_drop = ["is_longest", "BasePeakID"]

alt_tts = alt_tts.drop(columns=cols_to_drop, errors="ignore")
std_tts = std_tts.drop(columns=cols_to_drop, errors="ignore")

# === Save Results ===
alt_tts.to_csv(alt_tts_path, sep="\t", index=False)
std_tts.to_csv(std_tts_path, sep="\t", index=False)

print(f"Table 1 (alt-TTS): {len(alt_tts)} rows → saved to:\n{alt_tts_path}")
print(f"Table 2 (std-TTS):  {len(std_tts)} rows → saved to:\n{std_tts_path}")

Table 1 (alt-TTS): 106 rows → saved to:
/home/ldufour/Documents/Project_Yeast/Output/TTS/Alt-TTS_solve.csv
Table 2 (std-TTS):  4896 rows → saved to:
/home/ldufour/Documents/Project_Yeast/Output/TTS/Std-TTS_solve.csv
